<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/03_Missingness_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
# ============================================================
# AIR-LLM — NOTEBOOK 03
# 03.0 ENVIRONMENT & CONFIGURATION
# ============================================================

from pathlib import Path
import json
import hashlib
import gc
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("=" * 100)
print("AIR-LLM — NOTEBOOK 03")
print("MISSINGNESS EXPERIMENTATION")
print("=" * 100)

# ------------------------------------------------------------
# Canonical Google Drive project
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AIR_LLM_Research"
)

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"AIR-LLM project directory not found:\n{PROJECT_ROOT}"
    )

# ------------------------------------------------------------
# Notebook 02 INPUTS
# ------------------------------------------------------------

NOTEBOOK_02_DIR = (
    PROJECT_ROOT / "data" / "notebook_02"
)

SPLIT_DIR = (
    NOTEBOOK_02_DIR / "splits"
)

# ------------------------------------------------------------
# Notebook 03 OUTPUTS
# ------------------------------------------------------------

NOTEBOOK_03_DIR = (
    PROJECT_ROOT / "data" / "notebook_03"
)

MASK_DIR = (
    NOTEBOOK_03_DIR / "masks"
)

GROUND_TRUTH_DIR = (
    NOTEBOOK_03_DIR / "ground_truth"
)

DIAGNOSTIC_DIR = (
    NOTEBOOK_03_DIR / "diagnostics"
)

METADATA_DIR = (
    NOTEBOOK_03_DIR / "metadata"
)

for path in [
    NOTEBOOK_03_DIR,
    MASK_DIR,
    GROUND_TRUTH_DIR,
    DIAGNOSTIC_DIR,
    METADATA_DIR
]:
    path.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# DATASETS
# ------------------------------------------------------------

DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]

TARGET_REGISTRY = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted"
}

# ------------------------------------------------------------
# EXPERIMENTAL DESIGN
# ------------------------------------------------------------

MECHANISMS = [
    "MCAR",
    "MAR",
    "MNAR_APPROXIMATION"
]

MISSINGNESS_RATES = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50
]

REPETITIONS = 5
MASTER_SEED = 42

# ------------------------------------------------------------
# Registry
# ------------------------------------------------------------

SCENARIO_REGISTRY_PATH = (
    METADATA_DIR / "scenario_registry.csv"
)

MANIFEST_PATH = (
    PROJECT_ROOT / "artifacts" / "notebook_03_manifest.json"
)

MANIFEST_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

print("\nPROJECT")
print("-" * 100)
print(f"Project root : {PROJECT_ROOT}")
print(f"Notebook 02  : {NOTEBOOK_02_DIR}")
print(f"Notebook 03  : {NOTEBOOK_03_DIR}")

print("\nEXPERIMENT DESIGN")
print("-" * 100)
print(f"Datasets     : {len(DATASETS)}")
print(f"Mechanisms   : {len(MECHANISMS)}")
print(f"Rates        : {MISSINGNESS_RATES}")
print(f"Repetitions  : {REPETITIONS}")
print(f"Total cases  : {len(DATASETS) * len(MECHANISMS) * len(MISSINGNESS_RATES) * REPETITIONS}")
print(f"Seed         : {MASTER_SEED}")

print("\nNotebook 03 environment initialized.")

AIR-LLM — NOTEBOOK 03
MISSINGNESS EXPERIMENTATION

PROJECT
----------------------------------------------------------------------------------------------------
Project root : /content/drive/MyDrive/AIR_LLM_Research
Notebook 02  : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_02
Notebook 03  : /content/drive/MyDrive/AIR_LLM_Research/data/notebook_03

EXPERIMENT DESIGN
----------------------------------------------------------------------------------------------------
Datasets     : 3
Mechanisms   : 3
Rates        : [0.1, 0.2, 0.3, 0.4, 0.5]
Repetitions  : 5
Total cases  : 225
Seed         : 42

Notebook 03 environment initialized.


In [21]:
# ============================================================
# CELL 03.1 — LOAD TRAINING DATA
# ============================================================

TRAIN_DATA = {}

print("=" * 100)
print("AIR-LLM — CELL 03.1")
print("LOADING NOTEBOOK 02 TRAINING DATA")
print("=" * 100)

for dataset_id in DATASETS:

    path = (
        SPLIT_DIR
        / dataset_id
        / f"{dataset_id}_train.csv"
    )

    if not path.is_file():
        raise FileNotFoundError(
            f"Training file not found:\n{path}"
        )

    df = pd.read_csv(path)

    if df.empty:
        raise ValueError(
            f"Training dataset is empty:\n{path}"
        )

    target = TARGET_REGISTRY[dataset_id]

    if target not in df.columns:
        raise ValueError(
            f"Target '{target}' not found in {dataset_id}"
        )

    TRAIN_DATA[dataset_id] = df

    print(
        f"{dataset_id:20s} | "
        f"Rows: {len(df):8d} | "
        f"Columns: {len(df.columns):3d} | "
        f"Target: {target}"
    )

print("\nAll training datasets loaded successfully.")

AIR-LLM — CELL 03.1
LOADING NOTEBOOK 02 TRAINING DATA
adult_income         | Rows:    22792 | Columns:  16 | Target: income
bank_marketing       | Rows:    31647 | Columns:  18 | Target: y
diabetes_130us       | Rows:    71236 | Columns:  49 | Target: readmitted

All training datasets loaded successfully.


In [22]:
# ============================================================
# CELL 03.1 — LOAD TRAINING DATA
# ============================================================

TRAIN_DATA = {}

print("=" * 100)
print("AIR-LLM — CELL 03.1")
print("LOADING NOTEBOOK 02 TRAINING DATA")
print("=" * 100)

for dataset_id in DATASETS:

    path = (
        SPLIT_DIR
        / dataset_id
        / f"{dataset_id}_train.csv"
    )

    if not path.is_file():
        raise FileNotFoundError(
            f"Training file not found:\n{path}"
        )

    df = pd.read_csv(path)

    if df.empty:
        raise ValueError(
            f"Training dataset is empty:\n{path}"
        )

    target = TARGET_REGISTRY[dataset_id]

    if target not in df.columns:
        raise ValueError(
            f"Target '{target}' not found in {dataset_id}"
        )

    TRAIN_DATA[dataset_id] = df

    print(
        f"{dataset_id:20s} | "
        f"Rows: {len(df):8d} | "
        f"Columns: {len(df.columns):3d} | "
        f"Target: {target}"
    )

print("\nAll training datasets loaded successfully.")

AIR-LLM — CELL 03.1
LOADING NOTEBOOK 02 TRAINING DATA
adult_income         | Rows:    22792 | Columns:  16 | Target: income
bank_marketing       | Rows:    31647 | Columns:  18 | Target: y
diabetes_130us       | Rows:    71236 | Columns:  49 | Target: readmitted

All training datasets loaded successfully.


In [23]:
# ============================================================
# CELL 03.3 — DEFINE MASKING ENGINE
# ============================================================

def stable_seed(dataset_id, mechanism, rate, repetition):
    """
    Deterministic scenario seed.
    """
    key = (
        f"{MASTER_SEED}|{dataset_id}|"
        f"{mechanism}|{rate:.4f}|{repetition}"
    )

    digest = hashlib.sha256(
        key.encode("utf-8")
    ).hexdigest()

    return int(digest[:8], 16)


def numeric_score(series):
    """
    Robust numeric score in [0, 1].
    """
    x = pd.to_numeric(
        series,
        errors="coerce"
    ).to_numpy(dtype=float)

    valid = np.isfinite(x)

    score = np.full(
        len(x),
        np.nan,
        dtype=np.float64
    )

    if valid.sum() == 0:
        return score

    values = x[valid]

    q01, q99 = np.nanpercentile(
        values,
        [1, 99]
    )

    if q99 <= q01:
        q01 = np.nanmin(values)
        q99 = np.nanmax(values)

    if q99 <= q01:
        score[valid] = 0.5
    else:
        clipped = np.clip(
            values,
            q01,
            q99
        )

        score[valid] = (
            clipped - q01
        ) / (q99 - q01)

    return score


def categorical_score(series):
    """
    Frequency-based categorical score in [0,1].
    """
    s = series.astype("object")

    frequencies = (
        s.value_counts(
            normalize=True,
            dropna=True
        )
    )

    score = (
        s.map(frequencies)
        .to_numpy(dtype=float)
    )

    valid = np.isfinite(score)

    if valid.sum() == 0:
        return score

    min_v = np.nanmin(score[valid])
    max_v = np.nanmax(score[valid])

    if max_v > min_v:
        score[valid] = (
            score[valid] - min_v
        ) / (max_v - min_v)
    else:
        score[valid] = 0.5

    return score


def choose_driver(df, target_feature, maskable_features):
    """
    Select an observed driver for MAR.
    Preference:
    1. numerical feature with strongest absolute correlation
    2. categorical feature
    3. first available feature
    """

    candidates = [
        c for c in maskable_features
        if c != target_feature
    ]

    if not candidates:
        return None

    target_numeric = pd.to_numeric(
        df[target_feature],
        errors="coerce"
    )

    if target_numeric.notna().sum() >= 20:

        correlations = []

        for c in candidates:

            candidate_numeric = pd.to_numeric(
                df[c],
                errors="coerce"
            )

            valid = (
                target_numeric.notna()
                &
                candidate_numeric.notna()
            )

            if valid.sum() >= 20:

                corr = target_numeric[
                    valid
                ].corr(
                    candidate_numeric[
                        valid
                    ]
                )

                if pd.notna(corr):
                    correlations.append(
                        (
                            c,
                            abs(float(corr))
                        )
                    )

        if correlations:
            correlations.sort(
                key=lambda x: x[1],
                reverse=True
            )

            return correlations[0][0]

    return candidates[0]


def select_exact_cells(scores, eligible, rate, rng):
    """
    Select exactly floor(rate * eligible_cells)
    cells using score-weighted ranking.
    """

    valid_indices = np.flatnonzero(
        eligible
    )

    n_eligible = len(valid_indices)

    if n_eligible == 0:
        return np.zeros(
            len(eligible),
            dtype=bool
        )

    n_missing = int(
        round(rate * n_eligible)
    )

    n_missing = min(
        max(n_missing, 0),
        n_eligible
    )

    if n_missing == 0:
        return np.zeros(
            len(eligible),
            dtype=bool
        )

    local_scores = scores[
        valid_indices
    ]

    random_noise = rng.random(
        n_eligible
    )

    # Stable random tie breaking
    order = np.lexsort(
        (
            random_noise,
            local_scores
        )
    )

    chosen_local = order[
        -n_missing:
    ]

    chosen = valid_indices[
        chosen_local
    ]

    result = np.zeros(
        len(eligible),
        dtype=bool
    )

    result[chosen] = True

    return result


def generate_missingness_mask(
    df,
    target,
    mechanism,
    rate,
    seed,
    maskable_features
):
    """
    Generate a boolean DataFrame.

    True  = value should be masked
    False = value remains observed
    """

    rng = np.random.default_rng(
        seed
    )

    mask = pd.DataFrame(
        False,
        index=df.index,
        columns=maskable_features
    )

    for feature in maskable_features:

        values = df[feature]

        eligible = values.notna().to_numpy()

        if not eligible.any():
            continue

        if mechanism == "MCAR":

            scores = np.zeros(
                len(df),
                dtype=float
            )

            # Random ordering gives MCAR
            scores = rng.random(
                len(df)
            )

        elif mechanism == "MAR":

            driver = choose_driver(
                df,
                feature,
                maskable_features
            )

            if driver is None:
                scores = rng.random(
                    len(df)
                )
            else:

                driver_values = df[
                    driver
                ]

                if pd.api.types.is_numeric_dtype(
                    driver_values
                ):
                    scores = numeric_score(
                        driver_values
                    )
                else:
                    scores = categorical_score(
                        driver_values
                    )

                # Missing driver values receive
                # neutral score.
                scores = np.nan_to_num(
                    scores,
                    nan=0.5
                )

                # Small random component prevents ties.
                scores = (
                    0.90 * scores
                    +
                    0.10 * rng.random(
                        len(df)
                    )
                )

        elif mechanism == "MNAR_APPROXIMATION":

            # Self-masking approximation:
            # missingness depends on the feature's
            # observed magnitude/frequency.

            if pd.api.types.is_numeric_dtype(
                values
            ):
                scores = numeric_score(
                    values
                )
            else:
                scores = categorical_score(
                    values
                )

            scores = np.nan_to_num(
                scores,
                nan=0.5
            )

            scores = (
                0.90 * scores
                +
                0.10 * rng.random(
                    len(df)
                )
            )

        else:
            raise ValueError(
                f"Unknown mechanism: {mechanism}"
            )

        selected = select_exact_cells(
            scores=scores,
            eligible=eligible,
            rate=rate,
            rng=rng
        )

        mask.loc[
            selected,
            feature
        ] = True

    return mask


print("Masking engine defined successfully.")

Masking engine defined successfully.


In [24]:
# ============================================================
# CELL 03.4 — MCAR GENERATION
# ============================================================

def save_mask(mask, path):
    """
    Save boolean mask using bit packing.
    """
    array = mask.to_numpy(
        dtype=np.bool_,
        copy=False
    )

    packed = np.packbits(
        array,
        axis=None
    )

    np.savez_compressed(
        path,
        data=packed,
        shape=np.array(
            array.shape,
            dtype=np.int64
        ),
        columns=np.array(
            mask.columns.astype(str),
            dtype=object
        )
    )


def load_mask(path):
    """
    Reconstruct mask from packed storage.
    """
    with np.load(
        path,
        allow_pickle=True
    ) as z:

        shape = tuple(
            z["shape"].tolist()
        )

        unpacked = np.unpackbits(
            z["data"]
        )[:np.prod(shape)]

        array = unpacked.reshape(
            shape
        ).astype(bool)

        columns = [
            str(x)
            for x in z["columns"]
        ]

    return pd.DataFrame(
        array,
        columns=columns
    )


MCAR_COUNT = 0

print("=" * 100)
print("AIR-LLM — CELL 03.4")
print("MCAR MASK GENERATION")
print("=" * 100)

for dataset_id in DATASETS:

    df = GROUND_TRUTH[
        dataset_id
    ]

    target = TARGET_REGISTRY[
        dataset_id
    ]

    features = [
        c for c in df.columns
        if c != target
    ]

    for rate in MISSINGNESS_RATES:

        for repetition in range(
            1,
            REPETITIONS + 1
        ):

            seed = stable_seed(
                dataset_id,
                "MCAR",
                rate,
                repetition
            )

            mask = generate_missingness_mask(
                df=df,
                target=target,
                mechanism="MCAR",
                rate=rate,
                seed=seed,
                maskable_features=features
            )

            scenario_id = (
                f"{dataset_id}__MCAR__"
                f"r{int(rate*100):02d}__"
                f"rep{repetition}"
            )

            path = (
                MASK_DIR
                / f"{scenario_id}.npz"
            )

            save_mask(
                mask,
                path
            )

            MCAR_COUNT += 1

            del mask

    gc.collect()

print(
    f"MCAR masks generated: {MCAR_COUNT}"
)

AIR-LLM — CELL 03.4
MCAR MASK GENERATION
MCAR masks generated: 75


In [25]:
# ============================================================
# CELL 03.5 — MAR GENERATION
# ============================================================

MAR_COUNT = 0

print("=" * 100)
print("AIR-LLM — CELL 03.5")
print("MAR MASK GENERATION")
print("=" * 100)

for dataset_id in DATASETS:

    df = GROUND_TRUTH[
        dataset_id
    ]

    target = TARGET_REGISTRY[
        dataset_id
    ]

    features = [
        c for c in df.columns
        if c != target
    ]

    for rate in MISSINGNESS_RATES:

        for repetition in range(
            1,
            REPETITIONS + 1
        ):

            seed = stable_seed(
                dataset_id,
                "MAR",
                rate,
                repetition
            )

            mask = generate_missingness_mask(
                df=df,
                target=target,
                mechanism="MAR",
                rate=rate,
                seed=seed,
                maskable_features=features
            )

            scenario_id = (
                f"{dataset_id}__MAR__"
                f"r{int(rate*100):02d}__"
                f"rep{repetition}"
            )

            path = (
                MASK_DIR
                / f"{scenario_id}.npz"
            )

            save_mask(
                mask,
                path
            )

            MAR_COUNT += 1

            del mask

    gc.collect()

print(
    f"MAR masks generated: {MAR_COUNT}"
)

AIR-LLM — CELL 03.5
MAR MASK GENERATION
MAR masks generated: 75


In [26]:
# ============================================================
# CELL 03.6 — MNAR APPROXIMATION
# ============================================================

MNAR_COUNT = 0

print("=" * 100)
print("AIR-LLM — CELL 03.6")
print("MNAR APPROXIMATION MASK GENERATION")
print("=" * 100)

for dataset_id in DATASETS:

    df = GROUND_TRUTH[
        dataset_id
    ]

    target = TARGET_REGISTRY[
        dataset_id
    ]

    features = [
        c for c in df.columns
        if c != target
    ]

    for rate in MISSINGNESS_RATES:

        for repetition in range(
            1,
            REPETITIONS + 1
        ):

            seed = stable_seed(
                dataset_id,
                "MNAR_APPROXIMATION",
                rate,
                repetition
            )

            mask = generate_missingness_mask(
                df=df,
                target=target,
                mechanism="MNAR_APPROXIMATION",
                rate=rate,
                seed=seed,
                maskable_features=features
            )

            scenario_id = (
                f"{dataset_id}__MNAR_APPROXIMATION__"
                f"r{int(rate*100):02d}__"
                f"rep{repetition}"
            )

            path = (
                MASK_DIR
                / f"{scenario_id}.npz"
            )

            save_mask(
                mask,
                path
            )

            MNAR_COUNT += 1

            del mask

    gc.collect()

print(
    f"MNAR approximation masks generated: {MNAR_COUNT}"
)

AIR-LLM — CELL 03.6
MNAR APPROXIMATION MASK GENERATION
MNAR approximation masks generated: 75


In [27]:
# ============================================================
# CELL 03.7 — MISSINGNESS SCENARIO REGISTRY
# ============================================================

SCENARIO_ROWS = []

for dataset_id in DATASETS:

    for mechanism in MECHANISMS:

        for rate in MISSINGNESS_RATES:

            for repetition in range(
                1,
                REPETITIONS + 1
            ):

                seed = stable_seed(
                    dataset_id,
                    mechanism,
                    rate,
                    repetition
                )

                scenario_id = (
                    f"{dataset_id}__"
                    f"{mechanism}__"
                    f"r{int(rate*100):02d}__"
                    f"rep{repetition}"
                )

                mask_path = (
                    MASK_DIR
                    / f"{scenario_id}.npz"
                )

                SCENARIO_ROWS.append({
                    "scenario_id": scenario_id,
                    "dataset_id": dataset_id,
                    "mechanism": mechanism,
                    "requested_rate": rate,
                    "repetition": repetition,
                    "seed": seed,
                    "mask_path": str(mask_path)
                })

SCENARIO_REGISTRY_DF = pd.DataFrame(
    SCENARIO_ROWS
)

SCENARIO_REGISTRY_DF.to_csv(
    SCENARIO_REGISTRY_PATH,
    index=False
)

print("=" * 100)
print("SCENARIO REGISTRY")
print("=" * 100)

print(
    f"Scenarios registered: "
    f"{len(SCENARIO_REGISTRY_DF)}"
)

print(
    f"Expected scenarios  : "
    f"{len(DATASETS) * len(MECHANISMS) * len(MISSINGNESS_RATES) * REPETITIONS}"
)

display(
    SCENARIO_REGISTRY_DF.head(10)
)

SCENARIO REGISTRY
Scenarios registered: 225
Expected scenarios  : 225


,scenario_id,dataset_id,mechanism,requested_rate,repetition,seed,mask_path
0,adult_income__MCAR__r10__rep1,adult_income,MCAR,0.1,1,3920181198,/content/drive/MyDrive/AIR_LLM_Research/data/n...
1,adult_income__MCAR__r10__rep2,adult_income,MCAR,0.1,2,3719379823,/content/drive/MyDrive/AIR_LLM_Research/data/n...
2,adult_income__MCAR__r10__rep3,adult_income,MCAR,0.1,3,224306034,/content/drive/MyDrive/AIR_LLM_Research/data/n...
3,adult_income__MCAR__r10__rep4,adult_income,MCAR,0.1,4,1692051507,/content/drive/MyDrive/AIR_LLM_Research/data/n...
4,adult_income__MCAR__r10__rep5,adult_income,MCAR,0.1,5,3366700048,/content/drive/MyDrive/AIR_LLM_Research/data/n...
5,adult_income__MCAR__r20__rep1,adult_income,MCAR,0.2,1,2767163,/content/drive/MyDrive/AIR_LLM_Research/data/n...
6,adult_income__MCAR__r20__rep2,adult_income,MCAR,0.2,2,2740139601,/content/drive/MyDrive/AIR_LLM_Research/data/n...
7,adult_income__MCAR__r20__rep3,adult_income,MCAR,0.2,3,282536738,/content/drive/MyDrive/AIR_LLM_Research/data/n...
8,adult_income__MCAR__r20__rep4,adult_income,MCAR,0.2,4,1164937861,/content/drive/MyDrive/AIR_LLM_Research/data/n...
9,adult_income__MCAR__r20__rep5,adult_income,MCAR,0.2,5,3663661479,/content/drive/MyDrive/AIR_LLM_Research/data/n...


In [28]:
# ============================================================
# CELL 03.8 — REPEATED MASKING VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — CELL 03.8")
print("REPEATED MASKING VALIDATION")
print("=" * 100)

REPEATED_ROWS = []

for scenario in SCENARIO_REGISTRY_DF.itertuples(
    index=False
):

    dataset_id = scenario.dataset_id

    df = GROUND_TRUTH[
        dataset_id
    ]

    mask_path = Path(
        scenario.mask_path
    )

    if not mask_path.is_file():
        raise FileNotFoundError(
            f"Mask not found:\n{mask_path}"
        )

    mask = load_mask(
        mask_path
    )

    eligible = (
        ~df[
            mask.columns
        ].isna()
    ).to_numpy()

    mask_array = mask.to_numpy(
        dtype=bool
    )

    eligible_count = int(
        eligible.sum()
    )

    missing_count = int(
        mask_array.sum()
    )

    achieved_rate = (
        missing_count
        /
        max(
            eligible_count,
            1
        )
    )

    REPEATED_ROWS.append({
        "scenario_id": scenario.scenario_id,
        "dataset_id": dataset_id,
        "mechanism": scenario.mechanism,
        "requested_rate": scenario.requested_rate,
        "repetition": scenario.repetition,
        "seed": scenario.seed,
        "eligible_cells": eligible_count,
        "missing_cells": missing_count,
        "achieved_rate": achieved_rate,
        "rate_error": abs(
            achieved_rate
            -
            scenario.requested_rate
        )
    })

    del mask
    del mask_array
    del eligible

REPEATED_MASKING_DF = pd.DataFrame(
    REPEATED_ROWS
)

REPEATED_MASKING_DF.to_csv(
    DIAGNOSTIC_DIR
    / "repeated_masking_summary.csv",
    index=False
)

display(
    REPEATED_MASKING_DF.head(15)
)

print(
    f"Validated scenarios: "
    f"{len(REPEATED_MASKING_DF)}"
)

AIR-LLM — CELL 03.8
REPEATED MASKING VALIDATION


,scenario_id,dataset_id,mechanism,requested_rate,repetition,seed,eligible_cells,missing_cells,achieved_rate,rate_error
0,adult_income__MCAR__r10__rep1,adult_income,MCAR,0.1,1,3920181198,338917,33889,0.099992,0.000008
1,adult_income__MCAR__r10__rep2,adult_income,MCAR,0.1,2,3719379823,338917,33889,0.099992,0.000008
2,adult_income__MCAR__r10__rep3,adult_income,MCAR,0.1,3,224306034,338917,33889,0.099992,0.000008
3,adult_income__MCAR__r10__rep4,adult_income,MCAR,0.1,4,1692051507,338917,33889,0.099992,0.000008
4,adult_income__MCAR__r10__rep5,adult_income,MCAR,0.1,5,3366700048,338917,33889,0.099992,0.000008
5,adult_income__MCAR__r20__rep1,adult_income,MCAR,0.2,1,2767163,338917,67778,0.199984,0.000016
6,adult_income__MCAR__r20__rep2,adult_income,MCAR,0.2,2,2740139601,338917,67778,0.199984,0.000016
7,adult_income__MCAR__r20__rep3,adult_income,MCAR,0.2,3,282536738,338917,67778,0.199984,0.000016
8,adult_income__MCAR__r20__rep4,adult_income,MCAR,0.2,4,1164937861,338917,67778,0.199984,0.000016
9,adult_income__MCAR__r20__rep5,adult_income,MCAR,0.2,5,3663661479,338917,67778,0.199984,0.000016


Validated scenarios: 225


In [29]:
# ============================================================
# CELL 03.9 — MASK REPRODUCIBILITY
# ============================================================

print("=" * 100)
print("AIR-LLM — CELL 03.9")
print("MASK REPRODUCIBILITY VALIDATION")
print("=" * 100)

REPRO_ROWS = []

for scenario in SCENARIO_REGISTRY_DF.itertuples(
    index=False
):

    df = GROUND_TRUTH[
        scenario.dataset_id
    ]

    target = TARGET_REGISTRY[
        scenario.dataset_id
    ]

    features = [
        c for c in df.columns
        if c != target
    ]

    mask_a = generate_missingness_mask(
        df=df,
        target=target,
        mechanism=scenario.mechanism,
        rate=scenario.requested_rate,
        seed=scenario.seed,
        maskable_features=features
    )

    mask_b = generate_missingness_mask(
        df=df,
        target=target,
        mechanism=scenario.mechanism,
        rate=scenario.requested_rate,
        seed=scenario.seed,
        maskable_features=features
    )

    identical = np.array_equal(
        mask_a.to_numpy(
            dtype=bool
        ),
        mask_b.to_numpy(
            dtype=bool
        )
    )

    REPRO_ROWS.append({
        "scenario_id": scenario.scenario_id,
        "dataset_id": scenario.dataset_id,
        "mechanism": scenario.mechanism,
        "rate": scenario.requested_rate,
        "repetition": scenario.repetition,
        "reproducible": identical
    })

    del mask_a
    del mask_b

REPRODUCIBILITY_DF = pd.DataFrame(
    REPRO_ROWS
)

REPRODUCIBILITY_DF.to_csv(
    DIAGNOSTIC_DIR
    / "mask_reproducibility.csv",
    index=False
)

print(
    "Reproducibility rate:",
    REPRODUCIBILITY_DF[
        "reproducible"
    ].mean()
)

if not REPRODUCIBILITY_DF[
    "reproducible"
].all():

    raise RuntimeError(
        "Mask reproducibility validation failed."
    )

print("Mask reproducibility VERIFIED.")

AIR-LLM — CELL 03.9
MASK REPRODUCIBILITY VALIDATION
Reproducibility rate: 1.0
Mask reproducibility VERIFIED.


In [30]:
# ============================================================
# CELL 03.10 — GROUND-TRUTH STORAGE
# ============================================================

print("=" * 100)
print("AIR-LLM — CELL 03.10")
print("GROUND-TRUTH STORAGE")
print("=" * 100)

GROUND_TRUTH_INDEX = []

for dataset_id in DATASETS:

    df = GROUND_TRUTH[
        dataset_id
    ]

    path = (
        GROUND_TRUTH_DIR
        / f"{dataset_id}_train_ground_truth.csv"
    )

    if not path.exists():

        df.to_csv(
            path,
            index=False
        )

    digest = hashlib.sha256()

    for chunk in pd.read_csv(
        path,
        chunksize=10000
    ):
        digest.update(
            pd.util.hash_pandas_object(
                chunk,
                index=True
            ).values.tobytes()
        )

    GROUND_TRUTH_INDEX.append({
        "dataset_id": dataset_id,
        "path": str(path),
        "rows": len(df),
        "columns": len(df.columns),
        "sha256": digest.hexdigest()
    })

GROUND_TRUTH_INDEX_DF = pd.DataFrame(
    GROUND_TRUTH_INDEX
)

GROUND_TRUTH_INDEX_DF.to_csv(
    METADATA_DIR
    / "ground_truth_index.csv",
    index=False
)

display(
    GROUND_TRUTH_INDEX_DF
)

AIR-LLM — CELL 03.10
GROUND-TRUTH STORAGE


,dataset_id,path,rows,columns,sha256
0,adult_income,/content/drive/MyDrive/AIR_LLM_Research/data/n...,22792,16,94aa3b8744263dd8e27b4ae9b10cab934c16f654447edd...
1,bank_marketing,/content/drive/MyDrive/AIR_LLM_Research/data/n...,31647,18,573dbd142de81aea86218869c09c6c7a088187b94c73f6...
2,diabetes_130us,/content/drive/MyDrive/AIR_LLM_Research/data/n...,71236,49,9301d2e126c17b549eda7aadea2e67538c91fe54559618...


In [31]:
# ============================================================
# CELL 03.11 — FEATURE MISSING COUNTS
# ============================================================

FEATURE_MISSING_ROWS = []

for dataset_id in DATASETS:

    df = GROUND_TRUTH[
        dataset_id
    ]

    for feature in df.columns:

        missing_count = int(
            df[feature].isna().sum()
        )

        FEATURE_MISSING_ROWS.append({
            "dataset_id": dataset_id,
            "feature": feature,
            "missing_count": missing_count,
            "observed_count": (
                len(df) - missing_count
            )
        })

FEATURE_MISSING_COUNT_DF = pd.DataFrame(
    FEATURE_MISSING_ROWS
)

FEATURE_MISSING_COUNT_DF.to_csv(
    DIAGNOSTIC_DIR
    / "feature_missing_counts.csv",
    index=False
)

display(
    FEATURE_MISSING_COUNT_DF.head(20)
)

,dataset_id,feature,missing_count,observed_count
0,adult_income,__air_llm_row_id,0,22792
1,adult_income,age,0,22792
2,adult_income,workclass,1275,21517
3,adult_income,fnlwgt,0,22792
4,adult_income,education,0,22792
5,adult_income,education_num,0,22792
6,adult_income,marital_status,0,22792
7,adult_income,occupation,1281,21511
8,adult_income,relationship,0,22792
9,adult_income,race,0,22792


In [32]:
# ============================================================
# CELL 03.12 — MISSING PERCENTAGES
# ============================================================

MISSING_PERCENTAGE_DF = (
    FEATURE_MISSING_COUNT_DF.copy()
)

MISSING_PERCENTAGE_DF[
    "missing_percentage"
] = (
    MISSING_PERCENTAGE_DF[
        "missing_count"
    ]
    /
    (
        MISSING_PERCENTAGE_DF[
            "missing_count"
        ]
        +
        MISSING_PERCENTAGE_DF[
            "observed_count"
        ]
    )
    * 100
)

MISSING_PERCENTAGE_DF.to_csv(
    DIAGNOSTIC_DIR
    / "feature_missing_percentages.csv",
    index=False
)

display(
    MISSING_PERCENTAGE_DF.head(20)
)

,dataset_id,feature,missing_count,observed_count,missing_percentage
0,adult_income,__air_llm_row_id,0,22792,0.000000
1,adult_income,age,0,22792,0.000000
2,adult_income,workclass,1275,21517,5.594068
3,adult_income,fnlwgt,0,22792,0.000000
4,adult_income,education,0,22792,0.000000
5,adult_income,education_num,0,22792,0.000000
6,adult_income,marital_status,0,22792,0.000000
7,adult_income,occupation,1281,21511,5.620393
8,adult_income,relationship,0,22792,0.000000
9,adult_income,race,0,22792,0.000000


In [33]:
# ============================================================
# CELL 03.13 — ROW-LEVEL MISSINGNESS
# ============================================================

ROW_MISSINGNESS_ROWS = []

for dataset_id in DATASETS:

    df = GROUND_TRUTH[
        dataset_id
    ]

    missing_per_row = (
        df.isna()
        .sum(axis=1)
    )

    ROW_MISSINGNESS_ROWS.append({
        "dataset_id": dataset_id,
        "rows": len(df),
        "mean_missing_features": float(
            missing_per_row.mean()
        ),
        "median_missing_features": float(
            missing_per_row.median()
        ),
        "max_missing_features": int(
            missing_per_row.max()
        ),
        "rows_with_missingness": int(
            (missing_per_row > 0).sum()
        ),
        "rows_complete": int(
            (missing_per_row == 0).sum()
        )
    })

ROW_MISSINGNESS_DF = pd.DataFrame(
    ROW_MISSINGNESS_ROWS
)

ROW_MISSINGNESS_DF.to_csv(
    DIAGNOSTIC_DIR
    / "row_level_missingness.csv",
    index=False
)

display(
    ROW_MISSINGNESS_DF
)

,dataset_id,rows,mean_missing_features,median_missing_features,max_missing_features,rows_with_missingness,rows_complete
0,adult_income,22792,0.130002,0.0,3,1668,21124
1,bank_marketing,31647,0.000000,0.0,0,0,31647
2,diabetes_130us,71236,3.675585,4.0,7,71236,0


In [34]:
# ============================================================
# CELL 03.14 — MISSINGNESS ASSOCIATIONS
# ============================================================

ASSOCIATION_ROWS = []

for dataset_id in DATASETS:

    df = GROUND_TRUTH[
        dataset_id
    ]

    missing_matrix = (
        df.isna()
        .astype(np.int8)
    )

    corr = missing_matrix.corr(
        numeric_only=True
    )

    features = list(
        corr.columns
    )

    for i, f1 in enumerate(features):

        for f2 in features[i + 1:]:

            value = corr.loc[
                f1,
                f2
            ]

            if pd.notna(value):

                ASSOCIATION_ROWS.append({
                    "dataset_id": dataset_id,
                    "feature_1": f1,
                    "feature_2": f2,
                    "missingness_correlation": float(
                        value
                    )
                })

ASSOCIATION_DF = pd.DataFrame(
    ASSOCIATION_ROWS
)

ASSOCIATION_DF.to_csv(
    DIAGNOSTIC_DIR
    / "missingness_associations.csv",
    index=False
)

print(
    f"Association pairs: "
    f"{len(ASSOCIATION_DF)}"
)

display(
    ASSOCIATION_DF.head(20)
)

Association pairs: 39


,dataset_id,feature_1,feature_2,missingness_correlation
0,adult_income,workclass,occupation,0.997516
1,adult_income,workclass,native_country,-0.003990
2,adult_income,occupation,native_country,-0.004136
3,diabetes_130us,race,weight,-0.025729
4,diabetes_130us,race,payer_code,-0.051056
5,diabetes_130us,race,medical_specialty,0.010951
6,diabetes_130us,race,diag_1,0.010761
7,diabetes_130us,race,diag_2,0.016834
8,diabetes_130us,race,diag_3,0.022121
9,diabetes_130us,race,max_glu_serum,0.019323


In [35]:
# ============================================================
# CELL 03.15 — MISSINGNESS PATTERN ANALYSIS
# ============================================================

PATTERN_ROWS = []

for dataset_id in DATASETS:

    df = GROUND_TRUTH[
        dataset_id
    ]

    mask = (
        df.isna()
        .to_numpy(
            dtype=np.uint8
        )
    )

    # Hash each row's missingness pattern
    row_hashes = []

    for row in mask:

        row_hashes.append(
            hashlib.md5(
                row.tobytes()
            ).hexdigest()[:12]
        )

    counts = (
        pd.Series(
            row_hashes
        )
        .value_counts()
    )

    for pattern_hash, count in counts.items():

        PATTERN_ROWS.append({
            "dataset_id": dataset_id,
            "pattern_hash": pattern_hash,
            "row_count": int(count),
            "row_percentage": (
                float(count)
                / len(df)
                * 100
            )
        })

PATTERN_DF = pd.DataFrame(
    PATTERN_ROWS
)

PATTERN_DF.to_csv(
    DIAGNOSTIC_DIR
    / "natural_missingness_patterns.csv",
    index=False
)

display(
    PATTERN_DF.head(20)
)

del mask
del row_hashes
gc.collect()

,dataset_id,pattern_hash,row_count,row_percentage
0,adult_income,4ae71336e44b,21124,92.681643
1,adult_income,88de028e33c8,1255,5.506318
2,adult_income,2a902eb0793e,387,1.697964
3,adult_income,62d442da98e0,20,0.087750
4,adult_income,285d5fed06dc,6,0.026325
5,bank_marketing,ff035bff2dcf,31647,100.000000
6,diabetes_130us,b965eec6ed7a,17377,24.393565
7,diabetes_130us,73356248c6f9,14627,20.533157
8,diabetes_130us,4022daf5f153,11428,16.042450
9,diabetes_130us,0f75e7d87b18,8542,11.991128


129

In [36]:
# ============================================================
# CELL 03.16 — MCAR DIAGNOSTIC EVIDENCE
# ============================================================

MCAR_DIAGNOSTIC_ROWS = []

for scenario in SCENARIO_REGISTRY_DF.itertuples(
    index=False
):

    if scenario.mechanism != "MCAR":
        continue

    df = GROUND_TRUTH[
        scenario.dataset_id
    ]

    mask = load_mask(
        Path(scenario.mask_path)
    )

    mask_numeric = (
        mask.astype(np.int8)
    )

    feature_rates = (
        mask_numeric.mean()
    )

    dispersion = float(
        feature_rates.std()
    )

    MCAR_DIAGNOSTIC_ROWS.append({
        "scenario_id": scenario.scenario_id,
        "dataset_id": scenario.dataset_id,
        "rate": scenario.requested_rate,
        "repetition": scenario.repetition,
        "mean_feature_missing_rate": float(
            feature_rates.mean()
        ),
        "std_feature_missing_rate": dispersion
    })

    del mask
    del mask_numeric

MCAR_DIAGNOSTIC_DF = pd.DataFrame(
    MCAR_DIAGNOSTIC_ROWS
)

MCAR_DIAGNOSTIC_DF.to_csv(
    DIAGNOSTIC_DIR
    / "mcar_diagnostic_evidence.csv",
    index=False
)

display(
    MCAR_DIAGNOSTIC_DF.head(15)
)

,scenario_id,dataset_id,rate,repetition,mean_feature_missing_rate,std_feature_missing_rate
0,adult_income__MCAR__r10__rep1,adult_income,0.1,1,0.099125,0.001974
1,adult_income__MCAR__r10__rep2,adult_income,0.1,2,0.099125,0.001974
2,adult_income__MCAR__r10__rep3,adult_income,0.1,3,0.099125,0.001974
3,adult_income__MCAR__r10__rep4,adult_income,0.1,4,0.099125,0.001974
4,adult_income__MCAR__r10__rep5,adult_income,0.1,5,0.099125,0.001974
5,adult_income__MCAR__r20__rep1,adult_income,0.2,1,0.198251,0.003955
6,adult_income__MCAR__r20__rep2,adult_income,0.2,2,0.198251,0.003955
7,adult_income__MCAR__r20__rep3,adult_income,0.2,3,0.198251,0.003955
8,adult_income__MCAR__r20__rep4,adult_income,0.2,4,0.198251,0.003955
9,adult_income__MCAR__r20__rep5,adult_income,0.2,5,0.198251,0.003955


In [37]:
# ============================================================
# CELL 03.17 — MAR DIAGNOSTIC EVIDENCE
# ============================================================

MAR_DIAGNOSTIC_ROWS = []

for scenario in SCENARIO_REGISTRY_DF.itertuples(
    index=False
):

    if scenario.mechanism != "MAR":
        continue

    df = GROUND_TRUTH[
        scenario.dataset_id
    ]

    target = TARGET_REGISTRY[
        scenario.dataset_id
    ]

    mask = load_mask(
        Path(scenario.mask_path)
    )

    for feature in mask.columns:

        missing_indicator = (
            mask[feature]
            .astype(float)
        )

        candidates = [
            c for c in df.columns
            if c != feature
            and c != target
        ]

        if not candidates:
            continue

        best_assoc = 0.0

        for driver in candidates[:10]:

            driver_numeric = pd.to_numeric(
                df[driver],
                errors="coerce"
            )

            valid = (
                driver_numeric.notna()
            )

            if valid.sum() < 20:
                continue

            corr = (
                driver_numeric[
                    valid
                ]
                .corr(
                    missing_indicator[
                        valid
                    ]
                )
            )

            if pd.notna(corr):
                best_assoc = max(
                    best_assoc,
                    abs(float(corr))
                )

        MAR_DIAGNOSTIC_ROWS.append({
            "scenario_id": scenario.scenario_id,
            "dataset_id": scenario.dataset_id,
            "feature": feature,
            "rate": scenario.requested_rate,
            "repetition": scenario.repetition,
            "observed_dependency_evidence": best_assoc
        })

    del mask

MAR_DIAGNOSTIC_DF = pd.DataFrame(
    MAR_DIAGNOSTIC_ROWS
)

MAR_DIAGNOSTIC_DF.to_csv(
    DIAGNOSTIC_DIR
    / "mar_diagnostic_evidence.csv",
    index=False
)

display(
    MAR_DIAGNOSTIC_DF.head(20)
)

,scenario_id,dataset_id,feature,rate,repetition,observed_dependency_evidence
0,adult_income__MAR__r10__rep1,adult_income,__air_llm_row_id,0.1,1,0.688803
1,adult_income__MAR__r10__rep1,adult_income,age,0.1,1,0.689233
2,adult_income__MAR__r10__rep1,adult_income,workclass,0.1,1,0.499364
3,adult_income__MAR__r10__rep1,adult_income,fnlwgt,0.1,1,0.641345
4,adult_income__MAR__r10__rep1,adult_income,education,0.1,1,0.516221
5,adult_income__MAR__r10__rep1,adult_income,education_num,0.1,1,0.041192
6,adult_income__MAR__r10__rep1,adult_income,marital_status,0.1,1,0.515940
7,adult_income__MAR__r10__rep1,adult_income,occupation,0.1,1,0.499104
8,adult_income__MAR__r10__rep1,adult_income,relationship,0.1,1,0.516206
9,adult_income__MAR__r10__rep1,adult_income,race,0.1,1,0.515960


In [38]:
# ============================================================
# CELL 03.18 — MNAR DIAGNOSTIC EVIDENCE
# ============================================================

MNAR_DIAGNOSTIC_ROWS = []

for scenario in SCENARIO_REGISTRY_DF.itertuples(
    index=False
):

    if scenario.mechanism != "MNAR_APPROXIMATION":
        continue

    df = GROUND_TRUTH[
        scenario.dataset_id
    ]

    target = TARGET_REGISTRY[
        scenario.dataset_id
    ]

    mask = load_mask(
        Path(scenario.mask_path)
    )

    for feature in mask.columns:

        observed = df[
            feature
        ].notna()

        missing_indicator = (
            mask[feature]
        )

        if observed.sum() < 20:
            continue

        if pd.api.types.is_numeric_dtype(
            df[feature]
        ):

            values = pd.to_numeric(
                df[feature],
                errors="coerce"
            )

            valid = (
                values.notna()
            )

            corr = (
                values[valid]
                .corr(
                    missing_indicator[valid]
                    .astype(float)
                )
            )

            evidence = (
                abs(float(corr))
                if pd.notna(corr)
                else 0.0
            )

        else:

            # Compare missingness by
            # observed category frequency.
            groups = pd.DataFrame({
                "value": df[feature],
                "missing": missing_indicator
            }).dropna(
                subset=["value"]
            )

            if len(groups) > 0:

                rates = (
                    groups.groupby(
                        "value"
                    )["missing"]
                    .mean()
                )

                evidence = float(
                    rates.max()
                    -
                    rates.min()
                )

            else:
                evidence = 0.0

        MNAR_DIAGNOSTIC_ROWS.append({
            "scenario_id": scenario.scenario_id,
            "dataset_id": scenario.dataset_id,
            "feature": feature,
            "rate": scenario.requested_rate,
            "repetition": scenario.repetition,
            "self_masking_evidence": evidence
        })

    del mask

MNAR_DIAGNOSTIC_DF = pd.DataFrame(
    MNAR_DIAGNOSTIC_ROWS
)

MNAR_DIAGNOSTIC_DF.to_csv(
    DIAGNOSTIC_DIR
    / "mnar_approximation_diagnostic_evidence.csv",
    index=False
)

display(
    MNAR_DIAGNOSTIC_DF.head(20)
)

,scenario_id,dataset_id,feature,rate,repetition,self_masking_evidence
0,adult_income__MNAR_APPROXIMATION__r10__rep1,adult_income,__air_llm_row_id,0.1,1,0.515944
1,adult_income__MNAR_APPROXIMATION__r10__rep1,adult_income,age,0.1,1,0.641259
2,adult_income__MNAR_APPROXIMATION__r10__rep1,adult_income,workclass,0.1,1,0.135893
3,adult_income__MNAR_APPROXIMATION__r10__rep1,adult_income,fnlwgt,0.1,1,0.688822
4,adult_income__MNAR_APPROXIMATION__r10__rep1,adult_income,education,0.1,1,0.312749
5,adult_income__MNAR_APPROXIMATION__r10__rep1,adult_income,education_num,0.1,1,0.528562
6,adult_income__MNAR_APPROXIMATION__r10__rep1,adult_income,marital_status,0.1,1,0.218128
7,adult_income__MNAR_APPROXIMATION__r10__rep1,adult_income,occupation,0.1,1,0.398095
8,adult_income__MNAR_APPROXIMATION__r10__rep1,adult_income,relationship,0.1,1,0.247422
9,adult_income__MNAR_APPROXIMATION__r10__rep1,adult_income,race,0.1,1,0.116932


In [39]:
# ============================================================
# CELL 03.19 — NATURAL MISSINGNESS SUMMARY
# ============================================================

NATURAL_ROWS = []

for dataset_id in DATASETS:

    df = GROUND_TRUTH[
        dataset_id
    ]

    total_cells = df.size
    missing_cells = int(
        df.isna().sum().sum()
    )

    features_with_missing = int(
        df.isna()
        .any()
        .sum()
    )

    complete_rows = int(
        df.notna()
        .all(axis=1)
        .sum()
    )

    NATURAL_ROWS.append({
        "dataset_id": dataset_id,
        "rows": len(df),
        "columns": len(df.columns),
        "total_cells": total_cells,
        "natural_missing_cells": missing_cells,
        "natural_missing_percentage": (
            missing_cells
            / max(total_cells, 1)
            * 100
        ),
        "features_with_missingness": features_with_missing,
        "complete_rows": complete_rows,
        "incomplete_rows": (
            len(df)
            -
            complete_rows
        )
    })

NATURAL_MISSINGNESS_DF = pd.DataFrame(
    NATURAL_ROWS
)

NATURAL_MISSINGNESS_DF.to_csv(
    DIAGNOSTIC_DIR
    / "natural_missingness_summary.csv",
    index=False
)

display(
    NATURAL_MISSINGNESS_DF
)

,dataset_id,rows,columns,total_cells,natural_missing_cells,natural_missing_percentage,features_with_missingness,complete_rows,incomplete_rows
0,adult_income,22792,16,364672,2963,0.812511,3,21124,1668
1,bank_marketing,31647,18,569646,0,0.000000,0,31647,0
2,diabetes_130us,71236,49,3490564,261834,7.501195,9,0,71236


In [40]:
# ============================================================
# CELL 03.20 — FINAL VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — CELL 03.20")
print("FINAL NOTEBOOK 03 VALIDATION")
print("=" * 100)

expected_scenarios = (
    len(DATASETS)
    *
    len(MECHANISMS)
    *
    len(MISSINGNESS_RATES)
    *
    REPETITIONS
)

actual_scenarios = len(
    SCENARIO_REGISTRY_DF
)

assert actual_scenarios == expected_scenarios, (
    f"Scenario count mismatch: "
    f"{actual_scenarios} != {expected_scenarios}"
)

# ------------------------------------------------------------
# Mask existence
# ------------------------------------------------------------

missing_masks = []

for path in SCENARIO_REGISTRY_DF[
    "mask_path"
]:

    if not Path(path).is_file():
        missing_masks.append(path)

assert not missing_masks, (
    f"Missing mask files: {len(missing_masks)}"
)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

assert REPRODUCIBILITY_DF[
    "reproducible"
].all()

# ------------------------------------------------------------
# Rate validation
# ------------------------------------------------------------

MAX_RATE_ERROR = (
    REPEATED_MASKING_DF[
        "rate_error"
    ].max()
)

print(
    f"Expected scenarios : {expected_scenarios}"
)

print(
    f"Actual scenarios   : {actual_scenarios}"
)

print(
    f"Mask files         : "
    f"{len(list(MASK_DIR.glob('*.npz')))}"
)

print(
    f"Maximum rate error : "
    f"{MAX_RATE_ERROR:.8f}"
)

print(
    f"Reproducibility    : "
    f"{REPRODUCIBILITY_DF['reproducible'].mean()*100:.2f}%"
)

print("\nFINAL VALIDATION PASSED.")

AIR-LLM — CELL 03.20
FINAL NOTEBOOK 03 VALIDATION
Expected scenarios : 225
Actual scenarios   : 225
Mask files         : 225
Maximum rate error : 0.00001593
Reproducibility    : 100.00%

FINAL VALIDATION PASSED.


In [41]:
# ============================================================
# CELL 03.21 — DRIVE PERSISTENCE VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — CELL 03.21")
print("DRIVE PERSISTENCE VALIDATION")
print("=" * 100)

required_paths = [
    PROJECT_ROOT,
    NOTEBOOK_02_DIR,
    SPLIT_DIR,
    NOTEBOOK_03_DIR,
    MASK_DIR,
    GROUND_TRUTH_DIR,
    DIAGNOSTIC_DIR,
    METADATA_DIR
]

for path in required_paths:

    if not path.exists():
        raise FileNotFoundError(
            f"Required path missing:\n{path}"
        )

required_outputs = [
    SCENARIO_REGISTRY_PATH,
    DIAGNOSTIC_DIR / "repeated_masking_summary.csv",
    DIAGNOSTIC_DIR / "mask_reproducibility.csv",
    DIAGNOSTIC_DIR / "feature_missing_counts.csv",
    DIAGNOSTIC_DIR / "feature_missing_percentages.csv",
    DIAGNOSTIC_DIR / "row_level_missingness.csv",
    DIAGNOSTIC_DIR / "missingness_associations.csv",
    DIAGNOSTIC_DIR / "natural_missingness_patterns.csv",
    DIAGNOSTIC_DIR / "mcar_diagnostic_evidence.csv",
    DIAGNOSTIC_DIR / "mar_diagnostic_evidence.csv",
    DIAGNOSTIC_DIR / "mnar_approximation_diagnostic_evidence.csv",
    DIAGNOSTIC_DIR / "natural_missingness_summary.csv"
]

missing_outputs = [
    str(p)
    for p in required_outputs
    if not p.is_file()
]

if missing_outputs:
    raise RuntimeError(
        "Required Notebook 03 outputs are missing:\n"
        + "\n".join(missing_outputs)
    )

print("Project directory : VERIFIED")
print("Notebook 02 input  : VERIFIED")
print("Notebook 03 output : VERIFIED")
print("Diagnostic files   : VERIFIED")
print("Mask persistence   : VERIFIED")

print("\nDrive persistence validation PASSED.")

AIR-LLM — CELL 03.21
DRIVE PERSISTENCE VALIDATION
Project directory : VERIFIED
Notebook 02 input  : VERIFIED
Notebook 03 output : VERIFIED
Diagnostic files   : VERIFIED
Mask persistence   : VERIFIED

Drive persistence validation PASSED.


In [42]:
# ============================================================
# CELL 03.22 — NOTEBOOK 03 MANIFEST
# ============================================================

manifest = {
    "project": "AIR-LLM",
    "notebook": "03_Missingness_Experimentation",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "datasets": DATASETS,

    "targets": TARGET_REGISTRY,

    "mechanisms": MECHANISMS,

    "missingness_rates": MISSINGNESS_RATES,

    "repetitions": REPETITIONS,

    "master_seed": MASTER_SEED,

    "total_scenarios": int(
        len(SCENARIO_REGISTRY_DF)
    ),

    "input_directory": str(
        SPLIT_DIR
    ),

    "output_directory": str(
        NOTEBOOK_03_DIR
    ),

    "mask_directory": str(
        MASK_DIR
    ),

    "ground_truth_directory": str(
        GROUND_TRUTH_DIR
    ),

    "diagnostic_directory": str(
        DIAGNOSTIC_DIR
    ),

    "scenario_registry": str(
        SCENARIO_REGISTRY_PATH
    ),

    "mask_storage": "compressed_bitpacked_npz",

    "ground_truth_preserved": True,

    "target_excluded_from_controlled_masking": True,

    "natural_missingness_preserved": True,

    "validation": {
        "scenario_count": True,
        "mask_existence": True,
        "reproducibility": True,
        "rate_validation": True,
        "drive_persistence": True
    },

    "methodological_note": (
        "MNAR is represented as an approximation using "
        "self-masking behavior. It is not claimed to "
        "identify true MNAR from observed data."
    )
}

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )

print("=" * 100)
print("AIR-LLM — CELL 03.22")
print("NOTEBOOK 03 MANIFEST")
print("=" * 100)

print(
    f"Manifest saved:\n{MANIFEST_PATH}"
)

print("\nManifest created successfully.")

AIR-LLM — CELL 03.22
NOTEBOOK 03 MANIFEST
Manifest saved:
/content/drive/MyDrive/AIR_LLM_Research/artifacts/notebook_03_manifest.json

Manifest created successfully.


In [43]:
# ============================================================
# CELL 03.23 — FINAL NOTEBOOK STATUS
# ============================================================

print("\n" + "=" * 100)
print("AIR-LLM — NOTEBOOK 03 COMPLETE")
print("=" * 100)

print("\nEXPERIMENTAL DESIGN")
print("-" * 100)

print(
    f"Datasets          : {len(DATASETS)}"
)

print(
    f"Mechanisms        : {len(MECHANISMS)}"
)

print(
    f"Missingness rates : {len(MISSINGNESS_RATES)}"
)

print(
    f"Repetitions       : {REPETITIONS}"
)

print(
    f"Total scenarios   : "
    f"{len(SCENARIO_REGISTRY_DF)}"
)

print("\nVALIDATION")
print("-" * 100)

print(
    "Scenario registry     : PASSED"
)

print(
    "Mask persistence      : PASSED"
)

print(
    "Mask reproducibility  : PASSED"
)

print(
    "Missingness rates     : PASSED"
)

print(
    "Ground truth          : PRESERVED"
)

print(
    "Natural missingness   : PRESERVED"
)

print(
    "Target masking        : EXCLUDED"
)

print(
    "Drive persistence     : PASSED"
)

print("\nOUTPUTS")
print("-" * 100)

print(
    f"Scenario registry:\n"
    f"{SCENARIO_REGISTRY_PATH}"
)

print(
    f"Mask directory:\n"
    f"{MASK_DIR}"
)

print(
    f"Ground truth:\n"
    f"{GROUND_TRUTH_DIR}"
)

print(
    f"Diagnostics:\n"
    f"{DIAGNOSTIC_DIR}"
)

print(
    f"Manifest:\n"
    f"{MANIFEST_PATH}"
)

print("\n" + "=" * 100)
print("ALL NOTEBOOK 03 VALIDATIONS PASSED")
print("AIR-LLM MISSINGNESS EXPERIMENTS ARE READY")
print("NOTEBOOK 04 — DATASET & FEATURE PROFILING MAY BEGIN")
print("=" * 100)


AIR-LLM — NOTEBOOK 03 COMPLETE

EXPERIMENTAL DESIGN
----------------------------------------------------------------------------------------------------
Datasets          : 3
Mechanisms        : 3
Missingness rates : 5
Repetitions       : 5
Total scenarios   : 225

VALIDATION
----------------------------------------------------------------------------------------------------
Scenario registry     : PASSED
Mask persistence      : PASSED
Mask reproducibility  : PASSED
Missingness rates     : PASSED
Ground truth          : PRESERVED
Natural missingness   : PRESERVED
Target masking        : EXCLUDED
Drive persistence     : PASSED

OUTPUTS
----------------------------------------------------------------------------------------------------
Scenario registry:
/content/drive/MyDrive/AIR_LLM_Research/data/notebook_03/metadata/scenario_registry.csv
Mask directory:
/content/drive/MyDrive/AIR_LLM_Research/data/notebook_03/masks
Ground truth:
/content/drive/MyDrive/AIR_LLM_Research/data/notebook_0